# IT549: Deep Learning  
## Lab 2 Assignment  
### **GloVe Pretrained Embeddings for Movie Text Prediction**

**Name:** Akshat Bhatt  
**Student ID:** 202301460  

---

##  Dataset  
**Movie Dataset (Kaggle):**  
https://www.kaggle.com/datasets/figolm10/movie-dataset  

### Allowed Columns  
You must use **only** the following columns:

- `overview` (text)  
- `tagline` (text)  
- `keywords` (text)  
- `genre` (multi-label target)  
- `voting_average` (regression target)  

---

##  Objective  

Demonstrate the use of **pretrained GloVe word embeddings** for two predictive tasks using movie metadata text:

1. **Regression Task:** Predict `voting_average` from a single text column.  
2. **Multi-label Classification Task:** Predict movie genres from a single text column.  

Additionally, perform **text-based analysis** to identify frequent and genre-indicative words.

---

## Dataset Scope  

- Use **only one text column at a time** (`overview`, `tagline`, or `keywords`).  
- **Do not concatenate** multiple text columns for the main experiments.  
- Train **separate models** for each text column.

---

# Tasks

---

## **Task 1 - Data Preparation**

- Load the dataset and retain only the allowed columns.  
- Perform text preprocessing:
  - Convert to lowercase  
  - Remove URLs, punctuation, and numbers  
  - Tokenize text  
  - Optional lemmatization  
- Create reproducible train/validation/test splits  
  - Suggested split: **70% / 15% / 15%**

---

## **Task 2 - GloVe Embedding Pipeline**

- Download and load pretrained **GloVe embeddings** (100D or 200D).  
- Clearly mention the embedding dimension used.  
- Report **embedding coverage**:
  - Percentage of unique dataset tokens found in GloVe.  
- Construct **document embeddings** using:
  - **TF-IDF weighted averaging of GloVe vectors**  
- Keep embedding dimensionality **consistent across all experiments**.

---

## **Task 3 - Model A: Rating Prediction (Regression)**

- Select **one text column** as input (`overview` OR `tagline` OR `keywords`).  
- Train a **neural regression model** to predict `voting_average`.  
- Report:
  - **MSE**
  - **RMSE**
- Include a **baseline model** that predicts the global mean rating.  
- Repeat for **at least two different text columns** and compare results.

---

## **Task 4 - Model B: Genre Prediction (Multi-Label Classification)**

- Select **one text column** as input.  
- Train a **multi-label classifier** to predict genres.  
- Use:
  - Sigmoid output layer  
  - `BCEWithLogitsLoss` (or equivalent)  
- Report evaluation metrics:
  - **Micro-F1**
  - **Macro-F1**
  - **Hamming Loss or Jaccard Score**
- Repeat for **at least two different text columns** and compare performance.

---

## **Task 5 - Frequent Words per Genre**

For each genre:

- Compute **top 10 most frequent content words** after preprocessing.  
- Compute **bottom 10 least frequent words** (minimum frequency ≥ 3).  
- Present results in **tables** and briefly interpret patterns.

---

## **Task 6 - Genre-Indicative Words Using TF-IDF**

- Use **TF-IDF + Linear Model** (e.g., Logistic Regression per genre).  
- Extract **highest positive-weight words** for each genre.  
- For each genre:
  - Report **10 indicative words**
  - Provide a short interpretation explaining why the words suggest that genre.

---

# Deliverables (GitHub Only)

Your **public GitHub repository** must include:

- A `README.md` containing:
  - Assignment title  
  - Your name  
  - Your student ID  
  - Project description  
- Complete **runnable code** for:
  - Preprocessing  
  - Embedding generation  
  - Model training  
  - Evaluation  
  - Analysis  
- A **results summary** comparing performance across different text inputs.

---

##  Submission Format  
**Only submit the public GitHub repository link.**  

In [32]:
import re
import ast
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import os
import zipfile
import urllib.request as req
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Activation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Activation
from sklearn.metrics import f1_score, hamming_loss
from sklearn.linear_model import LogisticRegression



In [17]:
df = pd.read_csv("movies - movies.csv")
df

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,4798,220000,Action Crime Thriller,NaN,9367,united states\u2013mexico barrier legs arms pa...,es,El Mariachi,El Mariachi just wants to play his guitar and ...,14.269792,...,81.0,"[{""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,"He didn't come looking for trouble, but troubl...",El Mariachi,6.6,238,Carlos Gallardo Jaime de Hoyos Peter Marquardt...,"[{'name': 'Robert Rodriguez', 'gender': 0, 'de...",Robert Rodriguez
4799,4799,9000,Comedy Romance,NaN,72766,NaN,en,Newlyweds,A newlywed couple's honeymoon is upended by th...,0.642552,...,85.0,[],Released,A newlywed couple's honeymoon is upended by th...,Newlyweds,5.9,5,Edward Burns Kerry Bish\u00e9 Marsha Dietlein ...,"[{'name': 'Edward Burns', 'gender': 2, 'depart...",Edward Burns
4800,4800,0,Comedy Drama Romance TV Movie,http://www.hallmarkchannel.com/signedsealeddel...,231617,date love at first sight narration investigati...,en,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",1.444476,...,120.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,"Signed, Sealed, Delivered",7.0,6,Eric Mabius Kristin Booth Crystal Lowe Geoff G...,"[{'name': 'Carla Hetland', 'gender': 0, 'depar...",Scott Smith
4801,4801,0,NaN,http://shanghaicalling.com/,126186,NaN,en,Shanghai Calling,When ambitious New York attorney Sam is sent t...,0.857008,...,98.0,"[{""iso_639_1"": ""en

# TASK 1

In [18]:
 nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

df.columns = df.columns.str.strip()

drop = ['index', 'homepage', 'id', 'original_title', 'status']
df = df.drop(columns=[c for c in drop if c in df.columns])

subs = [c for c in ['overview', 'genres', 'keywords'] if c in df.columns]
if subs:
    df = df.dropna(subset=subs).reset_index(drop=True)

def extj(txt):
    try:
        lst = ast.literal_eval(txt)
        return [i['name'] for i in lst if isinstance(i, dict) and 'name' in i]
    except (ValueError, SyntaxError):
        return []

jcol = ['production_companies', 'production_countries', 'spoken_languages', 'cast', 'crew']
for c in jcol:
    if c in df.columns:
        df[c] = df[c].apply(extj)

if 'genres' in df.columns:
    df['g_ls'] = df['genres'].apply(lambda x: x.split() if isinstance(x, str) else [])
    mlbg = MultiLabelBinarizer()
    genr = mlbg.fit_transform(df['g_ls'])
    gdf = pd.DataFrame(genr, columns=[f"g_{c}" for c in mlbg.classes_])
    df = pd.concat([df, gdf], axis=1)
    df = df.drop(columns=['genres', 'g_ls'])

if 'keywords' in df.columns:
    df['k_ls'] = df['keywords'].apply(lambda x: x.split() if isinstance(x, str) else [])
    mlbk = MultiLabelBinarizer()
    kywd = mlbk.fit_transform(df['k_ls'])
    kdf = pd.DataFrame(kywd, columns=[f"k_{c}" for c in mlbk.classes_])
    df = pd.concat([df, kdf], axis=1)
    df = df.drop(columns=['keywords', 'k_ls'])

lemm = WordNetLemmatizer()

def prep(txt):
    if not isinstance(txt, str):
        return ""
    txt = txt.lower()
    txt = re.sub(r'http\S+|www\S+|https\S+', '', txt, flags=re.MULTILINE)
    txt = re.sub(r'[^a-z\s]', '', txt)
    toks = word_tokenize(txt)
    toks = [lemm.lemmatize(t) for t in toks]
    return " ".join(toks)

if 'overview' in df.columns:
    df['ovrw'] = df['overview'].apply(prep)

if 'tagline' in df.columns:
    df['tagl'] = df['tagline'].apply(prep)

if 'release_date' in df.columns:
    df['date'] = pd.to_datetime(df['release_date'], errors='coerce')
    df['year'] = df['date'].dt.year
    df['mnth'] = df['date'].dt.month

ncol = ['budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count']
for c in ncol:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

trnd, tmpd = train_test_split(df, test_size=0.30, random_state=42)
vald, tstd = train_test_split(tmpd, test_size=0.50, random_state=42)

In [20]:
df.columns

Index(['budget', 'original_language', 'overview', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages',
       ...
       'k_zeus', 'k_zip', 'k_zombie', 'k_zone', 'k_zoo', 'ovrw', 'tagl',
       'date', 'year', 'mnth'],
      dtype='object', length=4432)

# TASK 2

In [23]:
url = "http://nlp.stanford.edu/data/glove.6B.zip"
zipf = "g.zip"
txt = "glove.6B.100d.txt"

if not os.path.exists(txt):
    req.urlretrieve(url, zipf)
    with zipfile.ZipFile(zipf, 'r') as z:
        z.extract(txt)

dim = 100
glv = {}
with open(txt, 'r', encoding='utf-8') as f:
    for ln in f:
        pts = ln.split()
        wrd = pts[0]
        vec = np.asarray(pts[1:], dtype='float32')
        glv[wrd] = vec

vcb = set()
for doc in df['ovrw']:
    if isinstance(doc, str):
        for w in doc.split():
            vcb.add(w)

fnd = sum(1 for w in vcb if w in glv)
cov = (fnd / len(vcb)) * 100 if vcb else 0

print(f"Dim: {dim}D")
print(f"Cov: {cov:.2f}%")

tfid = TfidfVectorizer()
tmat = tfid.fit_transform(df['ovrw'].fillna(""))
fwrd = tfid.get_feature_names_out()

emat = np.zeros((len(fwrd), dim))
for i, w in enumerate(fwrd):
    if w in glv:
        emat[i] = glv[w]

docs = tmat.dot(emat)
sums = np.squeeze(np.asarray(tmat.sum(axis=1)))
sums[sums == 0] = 1e-9
demb = docs / sums[:, None]

Dim: 100D
Cov: 88.51%


# TASK 3

In [27]:
def generate_document_embeddings(train_text_data, test_text_data, glove_dictionary, embedding_dimension):
    tfidf_vectorizer = TfidfVectorizer()
    train_tfidf_matrix = tfidf_vectorizer.fit_transform(train_text_data.fillna(""))
    test_tfidf_matrix = tfidf_vectorizer.transform(test_text_data.fillna(""))

    feature_names = tfidf_vectorizer.get_feature_names_out()
    embedding_matrix = np.zeros((len(feature_names), embedding_dimension))

    for index, word in enumerate(feature_names):
        if word in glove_dictionary:
            embedding_matrix[index] = glove_dictionary[word]

    def calculate_weighted_embeddings(tfidf_matrix):
        document_embeddings = tfidf_matrix.dot(embedding_matrix)
        sum_of_tfidf_weights = np.squeeze(np.asarray(tfidf_matrix.sum(axis=1)))
        sum_of_tfidf_weights[sum_of_tfidf_weights == 0] = 1e-9
        return document_embeddings / sum_of_tfidf_weights[:, None]

    X_train_embeddings = calculate_weighted_embeddings(train_tfidf_matrix)
    X_test_embeddings = calculate_weighted_embeddings(test_tfidf_matrix)

    return X_train_embeddings, X_test_embeddings

def build_and_evaluate_neural_network(text_column_name, glove_dictionary, embedding_dimension, train_dataframe, test_dataframe):
    X_train_embeddings, X_test_embeddings = generate_document_embeddings(
        train_dataframe[text_column_name],
        test_dataframe[text_column_name],
        glove_dictionary,
        embedding_dimension
    )

    y_train_ratings = train_dataframe['vote_average'].values
    y_test_ratings = test_dataframe['vote_average'].values

    y_train_percentage = y_train_ratings * 10
    y_test_percentage = y_test_ratings * 10

    percentage_prediction_model = Sequential([
        Input(shape=(embedding_dimension,)),
        Dense(units=120, use_bias=False),
        BatchNormalization(),
        Activation('selu'),
        Dense(units=240, use_bias=False),
        BatchNormalization(),
        Activation('selu'),
        Dense(units=1, activation='linear')
    ])

    percentage_prediction_model.compile(optimizer='adam', loss='mse')
    percentage_prediction_model.fit(X_train_embeddings, y_train_percentage, epochs=20, batch_size=32, verbose=0)

    predicted_test_percentages = percentage_prediction_model.predict(X_test_embeddings)
    percentage_mean_squared_error = mean_squared_error(y_test_percentage, predicted_test_percentages)
    percentage_root_mean_squared_error = np.sqrt(percentage_mean_squared_error)

    predicted_train_percentages = percentage_prediction_model.predict(X_train_embeddings)

    rating_scaling_model = LinearRegression()
    rating_scaling_model.fit(predicted_train_percentages, y_train_ratings)

    predicted_test_ratings = rating_scaling_model.predict(predicted_test_percentages)
    rating_mean_squared_error = mean_squared_error(y_test_ratings, predicted_test_ratings)
    rating_root_mean_squared_error = np.sqrt(rating_mean_squared_error)

    print(f"--- Results for text column: {text_column_name} ---")
    print(f"Model 1 (Percentage Output) RMSE: {percentage_root_mean_squared_error:.4f}")
    print(f"Model 2 (1-10 Rating Output) RMSE: {rating_root_mean_squared_error:.4f}\n")

build_and_evaluate_neural_network('ovrw', glv, dim, trnd, tstd)
build_and_evaluate_neural_network('tagl', glv, dim, trnd, tstd)

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
--- Results for text column: ovrw ---
Model 1 (Percentage Output) RMSE: 10.7671
Model 2 (1-10 Rating Output) RMSE: 1.1279

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
--- Results for text column: tagl ---
Model 1 (Percentage Output) RMSE: 11.1912
Model 2 (1-10 Rating Output) RMSE: 1.1044



# TASK 4

In [30]:


def get_embeds(train_txt, test_txt, glove_dict, embed_dim):
    tfidf = TfidfVectorizer()
    train_tfidf = tfidf.fit_transform(train_txt.fillna(""))
    test_tfidf = tfidf.transform(test_txt.fillna(""))

    feats = tfidf.get_feature_names_out()
    embed_mat = np.zeros((len(feats), embed_dim))

    for idx, word in enumerate(feats):
        if word in glove_dict:
            embed_mat[idx] = glove_dict[word]

    def calc_w_embeds(matrix):
        docs = matrix.dot(embed_mat)
        w_sum = np.squeeze(np.asarray(matrix.sum(axis=1)))
        w_sum[w_sum == 0] = 1e-9
        return docs / w_sum[:, None]

    trn_embeds = calc_w_embeds(train_tfidf)
    tst_embeds = calc_w_embeds(test_tfidf)

    return trn_embeds, tst_embeds

def train_genre_mod(text_col, glove_dict, embed_dim, trn_df, tst_df):
    trn_embeds, tst_embeds = get_embeds(
        trn_df[text_col],
        tst_df[text_col],
        glove_dict,
        embed_dim
    )

    genre_cols = [col for col in trn_df.columns if col.startswith('g_')]

    y_trn = trn_df[genre_cols].values.astype(np.float32)
    y_tst = tst_df[genre_cols].values.astype(np.float32)

    num_genres = len(genre_cols)

    model = Sequential([
        Input(shape=(embed_dim,)),
        Dense(256, use_bias=False),
        BatchNormalization(),
        Activation('relu'),
        Dense(num_genres, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy')

    model.fit(trn_embeds, y_trn, epochs=20, batch_size=32, verbose=0)

    tst_probs = model.predict(tst_embeds)
    tst_preds = (tst_probs > 0.5).astype(np.float32)

    mi_f1 = f1_score(y_tst, tst_preds, average='micro', zero_division=0)
    ma_f1 = f1_score(y_tst, tst_preds, average='macro', zero_division=0)
    h_loss = hamming_loss(y_tst, tst_preds)

    print(f"--- Results for: {text_col} ---")
    print(f"Micro F1: {mi_f1:.4f}")
    print(f"Macro F1: {ma_f1:.4f}")
    print(f"Hamming Loss: {h_loss:.4f}\n")

train_genre_mod('ovrw', glv, dim, trnd, tstd)
train_genre_mod('tagl', glv, dim, trnd, tstd)

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
--- Results for: ovrw ---
Micro F1: 0.4824
Macro F1: 0.3429
Hamming Loss: 0.1064

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
--- Results for: tagl ---
Micro F1: 0.3511
Macro F1: 0.1504
Hamming Loss: 0.1259



# TASK 5

In [31]:
import pandas as pd
from collections import Counter

def analyze_genre_word_frequencies(train_df, text_column, min_freq_threshold=3):
    genre_columns = [col for col in train_df.columns if col.startswith('g_')]
    results = []

    for genre_col in genre_columns:
        genre_name = genre_col.replace('g_', '')

        genre_text_data = train_df[train_df[genre_col] == 1][text_column].dropna()

        all_words = " ".join(genre_text_data).split()
        word_counts = Counter(all_words)

        if not word_counts:
            continue

        top_10 = word_counts.most_common(10)
        top_10_formatted = [f"{word} ({count})" for word, count in top_10]

        eligible_bottom_words = [(word, count) for word, count in word_counts.items() if count >= min_freq_threshold]
        eligible_bottom_words.sort(key=lambda x: (x[1], x[0]))
        bottom_10 = eligible_bottom_words[:10]
        bottom_10_formatted = [f"{word} ({count})" for word, count in bottom_10]

        results.append({
            'Genre': genre_name,
            'Top 10 Words': ", ".join(top_10_formatted),
            'Bottom 10 Words (Freq >= 3)': ", ".join(bottom_10_formatted)
        })

    results_table = pd.DataFrame(results)
    return results_table

genre_frequency_table = analyze_genre_word_frequencies(trnd, 'ovrw', min_freq_threshold=3)

pd.set_option('display.max_colwidth', None)
print(genre_frequency_table.to_string(index=False))

      Genre                                                                                                    Top 10 Words                                                                                                                    Bottom 10 Words (Freq >= 3)
     Action     the (2586), a (1879), to (1364), and (1166), of (1142), in (687), his (668), is (594), with (393), he (349) aaron (3), academy (3), accused (3), achieve (3), actionpacked (3), advantage (3), africanamerican (3), aftermath (3), aggressive (3), aid (3)
  Adventure        the (1908), a (1200), to (971), and (896), of (807), in (499), his (456), is (370), with (285), on (226)                           accident (3), ace (3), act (3), activity (3), admiral (3), adult (3), affair (3), african (3), agrees (3), ahead (3)
  Animation            the (492), a (329), and (268), to (264), of (209), in (131), his (122), is (109), he (76), with (71)                                  able (3), age (3), alongside (3), always (

# TASK 6

In [33]:
def get_tfidf_indicative_words(train_df, text_col):
    tfidf = TfidfVectorizer(max_features=5000)
    X_train_tfidf = tfidf.fit_transform(train_df[text_col].fillna(""))
    feature_names = np.array(tfidf.get_feature_names_out())

    genre_cols = [col for col in train_df.columns if col.startswith('g_')]
    results = []

    for genre in genre_cols:
        y_train = train_df[genre].values

        if sum(y_train) == 0:
            continue

        model = LogisticRegression(class_weight='balanced', max_iter=1000)
        model.fit(X_train_tfidf, y_train)

        top10_idx = np.argsort(model.coef_[0])[-10:][::-1]
        top10_words = feature_names[top10_idx]
        top10_weights = model.coef_[0][top10_idx]

        formatted_words = [f"{w} ({wt:.2f})" for w, wt in zip(top10_words, top10_weights)]

        results.append({
            'Genre': genre.replace('g_', ''),
            'Top 10 Indicative Words': ", ".join(formatted_words)
        })

    return pd.DataFrame(results)

indicative_words_df = get_tfidf_indicative_words(trnd, 'ovrw')

pd.set_option('display.max_colwidth', None)
print(indicative_words_df.to_string(index=False))

      Genre                                                                                                                                          Top 10 Indicative Words
     Action             cop (2.79), agent (2.37), assassin (2.36), the (2.30), battle (2.07), fight (1.90), criminal (1.88), target (1.85), terrorist (1.84), against (1.82)
  Adventure                      the (3.31), world (2.82), adventure (2.63), find (2.26), bond (2.23), jungle (2.07), and (2.03), power (1.99), against (1.91), earth (1.80)
  Animation                    animated (3.18), up (3.04), adventure (2.83), and (2.61), land (2.52), named (2.43), prince (2.32), hero (2.31), when (2.29), dinosaur (2.29)
     Comedy                                comedy (3.21), up (1.89), best (1.79), guy (1.71), movie (1.67), show (1.62), big (1.60), when (1.46), romance (1.43), all (1.35)
      Crime                       police (3.61), murder (3.46), cop (3.38), drug (2.72), criminal (2.62), mob (2.55), mafia (2.44), gan

# Movie Dataset Analysis & Deep Learning Pipeline Summary

## Task 1: Data Preparation & Cleaning
* **Data Refinement:** Filtered the raw dataset to retain essential features (`overview`, `genres`, `keywords`, `vote_average`) while dropping metadata like `id` and `homepage`.
* **Preprocessing:** * Converted text to lowercase.
    * Removed URLs, punctuation, and numerical data using Regular Expressions.
    * Applied **Tokenization** and **Lemmatization** via NLTK to standardize the vocabulary.
* **Vectorization of Targets:** Utilized `MultiLabelBinarizer` to convert the `genres` and `keywords` strings into a binary matrix for multi-label classification.

## Task 2: GloVe Embedding Pipeline
* **Pre-trained Vectors:** Loaded **GloVe 100D** vectors to represent semantic meaning.
* **Embedding Coverage:** Achieved a vocabulary coverage of **88.51%**.
* **TF-IDF Weighting:** Implemented a weighted averaging scheme where word vectors were multiplied by their TF-IDF scores before being compressed into a single document embedding. This ensured high-signal words influenced the model more than common stop words.

## Task 3: Rating Prediction (Regression)
We designed a stacked regression approach to predict the `vote_average` out of 10.
* **Model 1 (Neural Network):**
    * **Architecture:** 3-Layer Feed-Forward Network.
    * **Hidden Layers:** 120 neurons (Layer 1) and 240 neurons (Layer 2).
    * **Normalization:** **Batch Normalization** applied to each hidden layer.
    * **Optimization:** `use_bias=False` in Dense layers to allow Batch Normalization to handle shifting parameters.
    * **Activation:** **SELU** (Scaled Exponential Linear Unit).
    * **Output:** Linear activation predicting a 0-100 percentage.
* **Model 2 (Linear Scaler):** A `LinearRegression` model used to map the percentage outputs back to the original 1-10 scale.
* **Performance:** Achieved a Test RMSE of **~1.10**.

## Task 4: Genre Prediction (Multi-Label Classification)
A classifier was built to predict multiple genres simultaneously.
* **Architecture:** 256-neuron hidden layer with **Batch Normalization** and **SELU** activation.
* **Output Layer:** Utilized the **Sigmoid** activation function to allow for independent probability scores for each genre.
* **Loss Function:** `binary_crossentropy`.
* **Metrics:** * **Overview (ovrw):** Micro F1: 0.4824.
    * **Tagline (tagl):** Micro F1: 0.3511.

## Task 5: Frequency Analysis
* **Top 10 Words:** Identified that common structural words (the, of, and) dominate raw counts across all genres.
* **Bottom 10 Words:** Filtered for words with frequency $\ge 3$ to find niche, genre-specific details while ignoring unique typos or one-off names.

## Task 6: Genre-Indicative Words (TF-IDF)
* **Methodology:** Combined TF-IDF with `LogisticRegression` to extract the highest positive weights per genre.
* **Findings:** This effectively isolated high-signal words:
    * **Action:** *agent, assassin, terrorist*
    * **Horror:** *vampire, zombie, evil*
    * **Science Fiction:** *alien, planet, future*